<div style="background:#123B63;color:white;padding:14px 18px;border-radius:6px">
<b>CSE 816 &mdash; Machine Learning Lab</b> &nbsp;&middot;&nbsp; Department of CSE, University of Chittagong<br>
<span style="font-size:90%">Module 2 &middot; Week 5 &middot; Part 3 of 4 &nbsp;&middot;&nbsp; 60 minutes</span>
</div>

# Derivatives and Back-propagation

`model.fit` runs gradient descent on a cost that depends on every weight in the network. This part
opens it up. You will compute derivatives by nudging, then by the chain rule on a computation
graph, then for a whole network at once &mdash; and check every number against TensorFlow's
automatic differentiation. With your own gradients you will train a network with no library at
all, and finally replace plain gradient descent with **Adam**, written from its five-line
definition.

**Companion theory lecture:** CSE 815, Week 5, Part 2 (Multiclass Classification and
Back-propagation), sections on Adam and on derivatives and computation graphs.

## What you will be able to do

1. Estimate a derivative numerically, and choose the step $\varepsilon$ sensibly.
2. Run a computation graph forward and backward by hand, and confirm it with `tf.GradientTape`.
3. Write vectorised back-propagation for a two-layer network, and verify it with a gradient
   check.
4. Train a network with your own forward pass, backward pass and update rule.
5. Implement Adam and compare it with gradient descent on a badly scaled cost.

---

## 1. Setup

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # hide TensorFlow's start-up chatter

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

np.set_printoptions(precision=4, suppress=True)
keras.utils.set_random_seed(816)      # seeds Python, NumPy and TensorFlow together

BLUE, TEAL, AMBER, ROSE, GRAY = "#123B63", "#1B7F79", "#C97B17", "#9B2C4B", "#4A5560"

print("NumPy", np.__version__, "  TensorFlow", tf.__version__)

tf.keras.backend.set_floatx("float64")   # this part compares gradients to many digits

The last line makes TensorFlow compute in float64, like NumPy, so that our gradients can be
compared with TensorFlow's to ten digits rather than six.

---

## 2. A derivative is the effect of a nudge

If increasing $w$ by a tiny $\varepsilon$ increases $J(w)$ by about $k\,\varepsilon$, then
$dJ/dw = k$. The lecture's example: $J = w^2$ at $w = 3$.

In [ ]:
J = lambda w: w ** 2
eps = 0.001
print(f"J(3)       = {J(3.0)}")
print(f"J(3.001)   = {J(3.0 + eps):.6f}")
print(f"rise / eps = {(J(3.0 + eps) - J(3.0)) / eps:.6f}   (exact derivative 2w = 6)")
assert np.isclose(J(3.0 + eps), 9.006001)

The estimate is $6.001$, not $6$: the one-sided difference
$\bigl(J(w+\varepsilon) - J(w)\bigr)/\varepsilon$ is off by an amount proportional to
$\varepsilon$. The **central difference** $\bigl(J(w+\varepsilon) - J(w-\varepsilon)\bigr)/2\varepsilon$
cancels that term, and its error shrinks like $\varepsilon^2$. The lecture's table, both ways:

In [ ]:
cases = [("w^2", lambda w: w ** 2, lambda w: 2 * w),
         ("w^3", lambda w: w ** 3, lambda w: 3 * w ** 2),
         ("w",   lambda w: w,      lambda w: 1.0),
         ("1/w", lambda w: 1 / w,  lambda w: -1 / w ** 2)]
w0, eps = 2.0, 0.001
print(f"{'J(w)':>5} {'exact':>8} {'one-sided':>11} {'central':>11}")
for name, f, df in cases:
    one = (f(w0 + eps) - f(w0)) / eps
    cen = (f(w0 + eps) - f(w0 - eps)) / (2 * eps)
    print(f"{name:>5} {df(w0):8.4f} {one:11.6f} {cen:11.6f}")
    assert abs(cen - df(w0)) < abs(one - df(w0)) + 1e-12

How small should $\varepsilon$ be? Smaller removes more of the approximation error, but below a
point **round-off** takes over: $J(w+\varepsilon)$ and $J(w)$ agree in almost every digit and
their difference is mostly noise. Measure it.

In [ ]:
f, df = cases[1][1], cases[1][2]          # J = w^3 at w = 2; exact derivative 12
epss = np.logspace(-14, -1, 53)
err_one = [abs((f(w0 + e) - f(w0)) / e - df(w0)) for e in epss]
err_cen = [abs((f(w0 + e) - f(w0 - e)) / (2 * e) - df(w0)) for e in epss]

fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.loglog(epss, err_one, c=BLUE, lw=2, label="one-sided")
ax.loglog(epss, err_cen, c=TEAL, lw=2, label="central")
ax.set_xlabel(r"step $\varepsilon$"); ax.set_ylabel("error in dJ/dw")
ax.set_title("Approximation error falls, then round-off rises", fontsize=10)
ax.legend(fontsize=8)
plt.show()

best_one, best_cen = epss[np.argmin(err_one)], epss[np.argmin(err_cen)]
print(f"best eps, one-sided: {best_one:.0e}   error {min(err_one):.1e}")
print(f"best eps, central  : {best_cen:.0e}   error {min(err_cen):.1e}")
assert best_cen > best_one

The V shapes have their floor near $\varepsilon \approx 10^{-8}$ (one-sided) and
$\varepsilon$ of a few times $10^{-6}$ (central) in float64. In this lab every numerical check uses the
central difference with $\varepsilon = 10^{-6}$.

---

## 3. The computation graph

Linear regression on one example, $J = \frac{1}{2}(wx + b - y)^2$ with $w = 2$, $b = 8$,
$x = -2$, $y = 2$. Break it into one operation per node:

$$c = wx, \qquad a = c + b, \qquad d = a - y, \qquad J = \tfrac{1}{2}d^2.$$

**Forward**, left to right, storing each value. **Backward**, right to left: each node multiplies
the derivative arriving from its right by its own local derivative &mdash; the chain rule.

In [ ]:
w, b, x, y = 2.0, 8.0, -2.0, 2.0

# forward
c = w * x
a = c + b
d = a - y
J = 0.5 * d ** 2
print(f"forward : c = {c}, a = {a}, d = {d}, J = {J}")

# backward
dJ_dd = d                  # d(d^2 / 2)/dd = d
dJ_da = dJ_dd * 1.0        # d = a - y
dJ_dc = dJ_da * 1.0        # a = c + b
dJ_db = dJ_da * 1.0
dJ_dw = dJ_dc * x          # c = w x
print(f"backward: dJ/dd = {dJ_dd}, dJ/da = {dJ_da}, dJ/dc = {dJ_dc}, "
      f"dJ/db = {dJ_db}, dJ/dw = {dJ_dw}")

assert (c, a, d, J) == (-4.0, 4.0, 2.0, 2.0)
assert (dJ_dw, dJ_db) == (-4.0, 2.0)

Check the two parameter derivatives twice: by nudging, and with TensorFlow. `tf.GradientTape`
records the operations of the forward computation &mdash; the computation graph &mdash; and
`tape.gradient` runs back-propagation over it.

In [ ]:
def J_lin(w, b):
    return 0.5 * (w * x + b - y) ** 2


e = 1e-6
num_w = (J_lin(w + e, b) - J_lin(w - e, b)) / (2 * e)
num_b = (J_lin(w, b + e) - J_lin(w, b - e)) / (2 * e)
print(f"nudging   : dJ/dw = {num_w:.6f}, dJ/db = {num_b:.6f}")
print(f"lecture   : nudge w by 0.001 and J changes by {J_lin(w + 0.001, b) - J_lin(w, b):.6f}")

w_t, b_t = tf.Variable(w), tf.Variable(b)
with tf.GradientTape() as tape:
    J_t = 0.5 * (w_t * x + b_t - y) ** 2
g_w, g_b = tape.gradient(J_t, [w_t, b_t])
print(f"TensorFlow: dJ/dw = {g_w.numpy():.6f}, dJ/db = {g_b.numpy():.6f}")

assert np.allclose([num_w, num_b], [-4, 2]) and np.allclose([g_w, g_b], [-4, 2])

---

## 4. A larger example: two ReLU layers

One input, one unit per layer, ReLU in both, squared-error loss:

$$a^{[1]} = r(w^{[1]}x + b^{[1]}), \qquad a^{[2]} = r(w^{[2]}a^{[1]} + b^{[2]}), \qquad
  J = \tfrac{1}{2}(a^{[2]} - y)^2,$$

with $x = 1$, $y = 5$, $w^{[1]} = 2$, $b^{[1]} = 0$, $w^{[2]} = 3$, $b^{[2]} = 1$. The local
derivative of ReLU is $r'(z) = 1$ if $z > 0$ and $0$ otherwise.

In [ ]:
def two_relu(w1, b1, w2, b2, x=1.0, y=5.0):
    """Forward and backward pass. Returns J and the four parameter derivatives."""
    z1 = w1 * x + b1;   a1 = max(0.0, z1)
    z2 = w2 * a1 + b2;  a2 = max(0.0, z2)
    J = 0.5 * (a2 - y) ** 2

    dJ_da2 = a2 - y
    dJ_dz2 = dJ_da2 * (1.0 if z2 > 0 else 0.0)
    dJ_dw2 = dJ_dz2 * a1
    dJ_db2 = dJ_dz2
    dJ_da1 = dJ_dz2 * w2
    dJ_dz1 = dJ_da1 * (1.0 if z1 > 0 else 0.0)
    dJ_dw1 = dJ_dz1 * x
    dJ_db1 = dJ_dz1
    return J, np.array([dJ_dw1, dJ_db1, dJ_dw2, dJ_db2])


p0 = np.array([2.0, 0.0, 3.0, 1.0])            # w1, b1, w2, b2
J0, grads = two_relu(*p0)
print(f"J = {J0}")
print("back-prop (w1, b1, w2, b2):", grads)
assert J0 == 2.0 and np.array_equal(grads, [6, 6, 4, 2])

# The lecture's numerical check used a one-sided nudge of 0.001:
one_sided = [(two_relu(*(p0 + 0.001 * np.eye(4)[k]))[0] - J0) / 0.001 for k in range(4)]
print("one-sided nudges of 0.001 :", np.round(one_sided, 4))
assert np.allclose(one_sided, [6.0045, 6.0045, 4.0020, 2.0005], atol=5e-5)

params = [tf.Variable(v) for v in p0]
with tf.GradientTape() as tape:
    a1 = tf.nn.relu(params[0] * 1.0 + params[1])
    a2 = tf.nn.relu(params[2] * a1 + params[3])
    J_t = 0.5 * (a2 - 5.0) ** 2
tf_grads = np.array([g.numpy() for g in tape.gradient(J_t, params)])
print("TensorFlow               :", tf_grads)
assert np.allclose(tf_grads, grads)

Notice the reuse. $\partial J/\partial z^{[2]}$ was computed once and used three times: for
$w^{[2]}$, for $b^{[2]}$, and to continue backwards to $a^{[1]}$. That reuse is why one backward
pass costs about as much as one forward pass, however many parameters there are.

---

## 5. Back-propagation for a whole network

Now the lecture's $2 \to 3 \to 1$ network: ReLU hidden layer, sigmoid output, binary
cross-entropy cost over $m$ examples. Forward, in matrix form:

$$\mathbf{Z}^{[1]} = \mathbf{X}\mathbf{W}^{[1]} + \mathbf{b}^{[1]},\quad
  \mathbf{A}^{[1]} = r(\mathbf{Z}^{[1]}),\quad
  \mathbf{Z}^{[2]} = \mathbf{A}^{[1]}\mathbf{W}^{[2]} + \mathbf{b}^{[2]},\quad
  \mathbf{A}^{[2]} = g(\mathbf{Z}^{[2]}).$$

Backward, one line per node. The first line is the one you derived in Week 3: for a sigmoid
output with cross-entropy loss, $\partial L/\partial z = a - y$.

| quantity | formula | shape |
|---|---|---|
| $d\mathbf{Z}^{[2]}$ | $(\mathbf{A}^{[2]} - \mathbf{y})/m$ | $(m, 1)$ |
| $d\mathbf{W}^{[2]}$ | $\mathbf{A}^{[1]\top} d\mathbf{Z}^{[2]}$ | $(3, 1)$ |
| $d\mathbf{b}^{[2]}$ | column sums of $d\mathbf{Z}^{[2]}$ | $(1,)$ |
| $d\mathbf{A}^{[1]}$ | $d\mathbf{Z}^{[2]}\mathbf{W}^{[2]\top}$ | $(m, 3)$ |
| $d\mathbf{Z}^{[1]}$ | $d\mathbf{A}^{[1]} \odot [\mathbf{Z}^{[1]} > 0]$ | $(m, 3)$ |
| $d\mathbf{W}^{[1]}$ | $\mathbf{X}^\top d\mathbf{Z}^{[1]}$ | $(2, 3)$ |
| $d\mathbf{b}^{[1]}$ | column sums of $d\mathbf{Z}^{[1]}$ | $(3,)$ |

Every gradient has the same shape as the parameter it belongs to &mdash; a useful check.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def forward(X, P):
    Z1 = X @ P["W1"] + P["b1"]
    A1 = np.maximum(0.0, Z1)
    Z2 = A1 @ P["W2"] + P["b2"]
    A2 = sigmoid(Z2)
    return {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}


def cost(X, y, P):
    # -y log g(z) - (1 - y) log(1 - g(z)) simplifies to log(1 + e^z) - y z. np.logaddexp
    # evaluates log(e^0 + e^z) stably (Part 2), so a saturated output never gives log(0).
    z = forward(X, P)["Z2"][:, 0]
    return float(np.mean(np.logaddexp(0.0, z) - y * z))


def backward(X, y, P):
    m = len(y)
    c = forward(X, P)
    dZ2 = (c["A2"] - y.reshape(-1, 1)) / m
    dW2 = c["A1"].T @ dZ2
    db2 = dZ2.sum(axis=0)
    dA1 = dZ2 @ P["W2"].T
    dZ1 = dA1 * (c["Z1"] > 0)
    dW1 = X.T @ dZ1
    db1 = dZ1.sum(axis=0)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}


P_lec = {"W1": np.array([[1., -3., 5.], [2., 4., -6.]]), "b1": np.array([-1., 1., 2.]),
         "W2": np.array([[2.], [-1.], [-3.]]),          "b2": np.array([0.5])}
X_demo = np.array([[1.0, 2.0], [-1.0, -1.0]])
y_demo = np.array([1.0, 0.0])

print(f"cost on the lecture's two examples: {cost(X_demo, y_demo, P_lec):.6f}")
grads = backward(X_demo, y_demo, P_lec)
for k, g in grads.items():
    print(f"d{k} {str(g.shape):7}\n{g}")
    assert g.shape == P_lec[k].shape
assert np.isclose(cost(X_demo, y_demo, P_lec), 0.039547, atol=5e-7)   # as in Part 1

Two sanity checks before trusting this. The third hidden unit is off ($z < 0$) for the first
example, and the first two are off for the second; their slices of $d\mathbf{Z}^{[1]}$ must be
exactly zero. And the numbers must agree with TensorFlow.

In [ ]:
c = forward(X_demo, P_lec)
dZ1 = ((c["A2"] - y_demo.reshape(-1, 1)) / 2) @ P_lec["W2"].T * (c["Z1"] > 0)
print("Z1 > 0:\n", c["Z1"] > 0, "\ndZ1:\n", dZ1)
assert dZ1[0, 2] == 0 and dZ1[1, 0] == 0 and dZ1[1, 1] == 0

net = Sequential([Input(shape=(2,)),
                  Dense(3, activation="relu", name="L1"),
                  Dense(1, activation="sigmoid", name="L2")])
net.get_layer("L1").set_weights([P_lec["W1"], P_lec["b1"]])
net.get_layer("L2").set_weights([P_lec["W2"], P_lec["b2"]])
bce = keras.losses.BinaryCrossentropy()

with tf.GradientTape() as tape:
    J_t = bce(y_demo.reshape(-1, 1), net(X_demo))
tf_grads = tape.gradient(J_t, net.trainable_variables)      # W1, b1, W2, b2 in order

for k, g_tf in zip(["W1", "b1", "W2", "b2"], tf_grads):
    print(f"{k}: largest |mine - TensorFlow| = {np.abs(grads[k] - g_tf.numpy()).max():.1e}")
    assert np.allclose(grads[k], g_tf.numpy(), atol=1e-6)

Agreement to about $10^{-16}$: the same arithmetic, in the same precision. But TensorFlow
and our `backward` could share a mistake in reasoning. A **gradient check** against central
differences is the independent test, so run it on all 13 parameters:

In [ ]:
def gradient_check(X, y, P, eps=1e-6):
    """Largest relative difference between back-prop and central differences."""
    g = backward(X, y, P)
    worst = 0.0
    for k in P:
        for idx in np.ndindex(P[k].shape):
            old = P[k][idx]
            P[k][idx] = old + eps; Jp = cost(X, y, P)
            P[k][idx] = old - eps; Jm = cost(X, y, P)
            P[k][idx] = old
            num = (Jp - Jm) / (2 * eps)
            worst = max(worst, abs(num - g[k][idx]) / max(1e-8, abs(num) + abs(g[k][idx])))
    return worst


rel = gradient_check(X_demo, y_demo, P_lec)
print(f"largest relative difference over 13 parameters: {rel:.1e}")
print("z1 of the second example:", forward(X_demo, P_lec)["Z1"][1])

A relative difference of $7 \times 10^{-4}$ is far too large for a correct implementation
&mdash; and yet the implementation is correct. Look at the second example's $\mathbf{z}^{[1]}$:
the middle unit has $z = 0$ **exactly**, the kink of ReLU. There the slope is 0 on the left and
1 on the right; the central difference averages them to $\frac{1}{2}$, while `backward` (and
TensorFlow) use 0. Neither is wrong: ReLU has no derivative at 0, and libraries simply pick
one side.

Hand-picked integer examples hit the kink; real data almost never does. Check again on random
inputs:

In [ ]:
rng = np.random.default_rng(0)
X_r = rng.normal(size=(20, 2))
y_r = rng.integers(0, 2, 20).astype(float)
rel = gradient_check(X_r, y_r, P_lec)
print(f"random inputs: largest relative difference {rel:.1e}")
assert rel < 1e-6
assert not np.any(forward(X_r, P_lec)["Z1"] == 0)

---

## 6. Training with your own gradients

`forward`, `backward` and the update $\theta \leftarrow \theta - \alpha\,\partial J/\partial\theta$
are all `fit` does for this network. Use them on a data set that needs a curved boundary: two
interleaved half-moons.

Initial weights matter. All-zero weights would make every hidden unit identical forever (they
would receive identical gradients). We start from small random values scaled by
$\sqrt{2/n_{\text{in}}}$, the usual choice for ReLU layers.

In [ ]:
from sklearn.datasets import make_moons

X_m, y_m = make_moons(n_samples=200, noise=0.2, random_state=816)
y_m = y_m.astype(float)


def init_params(n_hidden, seed=816):
    rng = np.random.default_rng(seed)
    return {"W1": rng.normal(size=(2, n_hidden)) * np.sqrt(2 / 2), "b1": np.zeros(n_hidden),
            "W2": rng.normal(size=(n_hidden, 1)) * np.sqrt(2 / n_hidden), "b2": np.zeros(1)}


def train(P, update, steps):
    """update(P, grads, t) changes P in place. Returns the cost after every step."""
    history = []
    for t in range(1, steps + 1):
        update(P, backward(X_m, y_m, P), t)
        history.append(cost(X_m, y_m, P))
    return history


def gd_update(alpha):
    def update(P, grads, t):
        for k in P:
            P[k] -= alpha * grads[k]
    return update


P_gd = init_params(8)
assert gradient_check(X_m, y_m, P_gd) < 1e-6        # check the gradients on this network too
J_start = cost(X_m, y_m, P_gd)
hist_gd = train(P_gd, gd_update(alpha=0.5), steps=3000)
acc_gd = float(((forward(X_m, P_gd)["A2"][:, 0] >= 0.5) == y_m).mean())
print(f"cost: {J_start:.4f} at the start, {hist_gd[-1]:.4f} after 3000 steps")
print(f"training accuracy: {acc_gd:.3f}")
assert hist_gd[-1] < J_start and acc_gd >= 0.9

In [ ]:
def plot_boundary(ax, P, title):
    g1, g2 = np.meshgrid(np.linspace(-1.6, 2.6, 250), np.linspace(-1.2, 1.7, 250))
    grid = np.c_[g1.ravel(), g2.ravel()]
    prob = forward(grid, P)["A2"][:, 0].reshape(g1.shape)
    ax.contourf(g1, g2, prob, levels=[0, 0.5, 1], colors=[BLUE, ROSE], alpha=0.15)
    ax.contour(g1, g2, prob, levels=[0.5], colors=[GRAY], linewidths=1.5)
    ax.scatter(*X_m[y_m == 0].T, c=BLUE, s=14, label="y = 0")
    ax.scatter(*X_m[y_m == 1].T, c=ROSE, marker="x", s=18, label="y = 1")
    ax.set_title(title, fontsize=10); ax.legend(fontsize=8)


fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
plot_boundary(axes[0], P_gd, "Your own back-prop + gradient descent")
axes[1].plot(hist_gd, c=BLUE, lw=2)
axes[1].set_yscale("log"); axes[1].set_xlabel("step"); axes[1].set_ylabel("cost J")
axes[1].set_title("Learning curve", fontsize=10)
plt.tight_layout()
plt.show()

No TensorFlow was involved in training. Each of the 8 ReLU units contributes a straight piece of
the boundary; together they bend around the moons.

---

## 7. Adam

### The problem it solves

One learning rate rarely suits every parameter. The lecture's example is
$J = w_1^2 + 10\,w_2^2$: steep in $w_2$, shallow in $w_1$. Gradient descent from $(-4, 1)$ must
pick an $\alpha$ small enough for $w_2$, and then crawls along $w_1$.

In [ ]:
def J_q(w):
    return w[0] ** 2 + 10 * w[1] ** 2


def grad_q(w):
    return np.array([2 * w[0], 20 * w[1]])


def run_gd(alpha, steps, w0=(-4.0, 1.0)):
    w = np.array(w0); path = [w.copy()]
    for _ in range(steps):
        w = w - alpha * grad_q(w); path.append(w.copy())
    return np.array(path)


slow, zigzag = run_gd(0.01, 200), run_gd(0.09, 200)
print("first steps, alpha = 0.01:", slow[:3].round(3).tolist())
print("first steps, alpha = 0.09:", zigzag[:3].round(3).tolist())
assert np.allclose(zigzag[1], [-3.28, -0.8]) and np.allclose(slow[1], [-3.92, 0.8])
print(f"\nJ after 200 steps:  alpha 0.01 -> {J_q(slow[-1]):.2e},   alpha 0.09 -> {J_q(zigzag[-1]):.2e}")
print("w2 is multiplied by 1 - 20 alpha each step: at alpha = 0.1 that is -1, so w2 bounces")
print("between +1 and -1 forever; above 0.1 it diverges.")

### The algorithm

For each parameter $w$, with gradient $d$ at step $t$:

$$m \leftarrow \beta_1 m + (1-\beta_1)\,d, \qquad v \leftarrow \beta_2 v + (1-\beta_2)\,d^2,$$
$$\hat m = \frac{m}{1-\beta_1^t}, \qquad \hat v = \frac{v}{1-\beta_2^t}, \qquad
  w \leftarrow w - \alpha\,\frac{\hat m}{\sqrt{\hat v} + \epsilon}.$$

$m$ is a running average of the gradient, $v$ of its square. The hats correct for both starting
at zero. When the gradient keeps its sign, $\hat m/\sqrt{\hat v} \approx \pm 1$ and every step is
about $\alpha$ long, **whatever the gradient's size**; when it flips sign, $\hat m$ averages
towards zero and the steps shrink. Defaults: $\beta_1 = 0.9$, $\beta_2 = 0.999$,
$\epsilon = 10^{-7}$.

In [ ]:
class Adam:
    def __init__(self, alpha=0.001, beta1=0.9, beta2=0.999, epsilon=1e-7):
        self.alpha, self.beta1, self.beta2, self.epsilon = alpha, beta1, beta2, epsilon
        self.m, self.v = {}, {}

    def step(self, key, w, d, t):
        """Return the updated value of parameter `key` with gradient d at step t (t >= 1)."""
        m = self.beta1 * self.m.get(key, 0.0) + (1 - self.beta1) * d
        v = self.beta2 * self.v.get(key, 0.0) + (1 - self.beta2) * d ** 2
        self.m[key], self.v[key] = m, v
        m_hat = m / (1 - self.beta1 ** t)
        v_hat = v / (1 - self.beta2 ** t)
        return w - self.alpha * m_hat / (np.sqrt(v_hat) + self.epsilon)


# First step by hand: m_hat = d and v_hat = d^2, so the step is alpha * sign(d).
opt = Adam(alpha=0.3)
w1 = opt.step("w", np.array([-4.0, 1.0]), grad_q(np.array([-4.0, 1.0])), t=1)
print("first Adam step from (-4, 1):", w1.round(6))
assert np.allclose(w1, [-3.7, 0.7], atol=1e-6)

# Check one update against Keras's implementation.
var = tf.Variable(np.array([-4.0, 1.0]))
keras.optimizers.Adam(learning_rate=0.3).apply_gradients([(tf.constant(grad_q(var.numpy())), var)])
assert np.allclose(var.numpy(), w1, atol=1e-6)
print("Keras's Adam takes the same first step.")

Both coordinates moved by exactly $0.3$, although the gradient in $w_2$ is 2.5 times larger
than in $w_1$ ($20$ against $-8$). That is the per-parameter scaling at work. Now compare
whole paths.

In [ ]:
def run_adam(alpha, steps, w0=(-4.0, 1.0)):
    opt, w = Adam(alpha=alpha), np.array(w0)
    path = [w.copy()]
    for t in range(1, steps + 1):
        w = opt.step("w", w, grad_q(w), t); path.append(w.copy())
    return np.array(path)


adam_path = run_adam(0.3, 200)


def steps_to(path, tol=1e-2):
    J = np.array([J_q(w) for w in path])
    return int(np.argmax(J < tol)) if (J < tol).any() else None


for name, path in [("GD, alpha 0.01", slow), ("GD, alpha 0.09", zigzag), ("Adam, alpha 0.3", adam_path)]:
    print(f"{name:16}: J < 0.01 after {steps_to(path)} steps")

g1, g2 = np.meshgrid(np.linspace(-4.5, 1, 200), np.linspace(-1.3, 1.3, 200))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for ax, (name, path, col) in zip(axes, [("GD, alpha = 0.01", slow, TEAL),
                                        ("GD, alpha = 0.09", zigzag, ROSE),
                                        ("Adam, alpha = 0.3", adam_path, AMBER)]):
    ax.contour(g1, g2, g1 ** 2 + 10 * g2 ** 2, levels=[1, 4, 9, 16], colors=[GRAY], alpha=0.5)
    ax.plot(*path[:25].T, "-o", c=col, ms=3, lw=1.2)
    ax.set_title(f"{name}: first 25 steps", fontsize=10); ax.set_xlabel("$w_1$")
axes[0].set_ylabel("$w_2$")
plt.tight_layout()
plt.show()

Adam heads along $w_1$ at full speed from the first step; momentum carries $w_2$ past zero and
it oscillates gently before settling. But the zig-zagging gradient descent got there first. Is
Adam worse?

Not once you ask the practical question: **which $\alpha$ would you have chosen?** Gradient
descent with $\alpha = 0.09$ works because $0.09$ sits just below the divergence threshold of
$0.1$, which we know only because we can differentiate this $J$ by hand. Sweep $\alpha$ for both:

In [ ]:
def steps_needed(path, tol=1e-2):
    J = np.array([J_q(w) for w in path])
    if not np.all(np.isfinite(J)) or J.max() > 1e6:
        return "diverges"
    return steps_to(path, tol) or "> 300"


print(f"{'alpha':>6} {'gradient descent':>18} {'Adam':>8}")
for alpha in [0.003, 0.01, 0.03, 0.09, 0.1, 0.11, 0.3, 1.0, 3.0]:
    with np.errstate(over="ignore", invalid="ignore"):
        gd_steps = steps_needed(run_gd(alpha, 300))
    print(f"{alpha:6g} {str(gd_steps):>18} {str(steps_needed(run_adam(alpha, 300))):>8}")

assert steps_needed(run_gd(0.11, 300)) == "diverges"
assert all(isinstance(steps_needed(run_adam(a, 300)), int) for a in [0.03, 0.1, 0.3, 1.0, 3.0])

Gradient descent needs $\alpha$ inside a narrow window, below $0.1$ here and best right at its
edge; a little above, it diverges. Adam reaches the minimum for every $\alpha$ from $0.03$ to $3$
&mdash; a hundredfold range &mdash; in 50 to 260 steps. That robustness, not raw speed on a
hand-tuned problem, is why Adam is the default optimiser.

### Adam on the moons network

Same network, same starting weights, same 3000 steps; only the update rule changes.

In [ ]:
def adam_update(alpha):
    opt = Adam(alpha=alpha)
    def update(P, grads, t):
        for k in P:
            P[k] = opt.step(k, P[k], grads[k], t)
    return update


P_adam = init_params(8)
hist_adam = train(P_adam, adam_update(alpha=0.01), steps=3000)
acc_adam = float(((forward(X_m, P_adam)["A2"][:, 0] >= 0.5) == y_m).mean())

for step in [100, 300, 1000, 3000]:
    print(f"step {step:4d}:  GD cost {hist_gd[step - 1]:.4f}   Adam cost {hist_adam[step - 1]:.4f}")
print(f"training accuracy:  GD {acc_gd:.3f}   Adam {acc_adam:.3f}")

fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.plot(hist_gd, c=BLUE, lw=2, label="gradient descent, alpha 0.5")
ax.plot(hist_adam, c=AMBER, lw=2, label="Adam, alpha 0.01")
ax.set_yscale("log"); ax.set_xlabel("step"); ax.set_ylabel("cost J"); ax.legend(fontsize=8)
plt.show()

On this small problem the two end up close. Gradient descent's $\alpha = 0.5$ was tuned for it;
Adam's $0.01$ is near its library default and was not. In the reference run Adam finishes with
the lower cost (`0.067` against `0.073`) and accuracy `0.975` against `0.965`. Exercise 4 asks you
to measure how each one degrades when $\alpha$ is chosen badly.

### Checkpoint 1

Lecture, descriptive question 10: for $J = \frac{1}{2}(wx + b - y)^2$ with $w = 1$, $b = 2$,
$x = 3$, $y = 1$, write the forward and backward passes of section 3 for these values, then
confirm $\partial J/\partial w$ and $\partial J/\partial b$ with central differences and with
`tf.GradientTape`.

*Expected:* $c = 3$, $a = 5$, $d = 4$, $J = 8$; $\partial J/\partial w = 12$,
$\partial J/\partial b = 4$.

In [ ]:
# Your code here

### Checkpoint 2

Lecture, descriptive question 12: in the two-ReLU-layer example of section 4, change
$w^{[2]}$ from $3$ to $-1$.

1. Use `two_relu` to compute $J$ and the four derivatives, and confirm them with
   `tf.GradientTape`.
2. Run 100 steps of gradient descent with $\alpha = 0.1$ from this point. How far does each
   parameter move?
3. Explain in two sentences what has happened to the second unit, and name the problem from
   Part 1.

*Expected:* $z^{[2]} = -1$, so $a^{[2]} = 0$ and $J = 12.5$; all four derivatives are `0`, and
the parameters do not move at all.

In [ ]:
# Your code here

---

## 8. Recap

- A derivative is the ratio of $J$'s change to a small nudge in $w$. Use central differences;
  $\varepsilon \approx 10^{-6}$ balances approximation error against round-off.
- On a computation graph, run forward storing every value, then backward multiplying local
  derivatives (the chain rule). For the lecture's graph, $\partial J/\partial w = -4$,
  $\partial J/\partial b = 2$; for the two-ReLU example, `6, 6, 4, 2`.
- Back-propagation for a dense network is one matrix line per node. Its gradients match
  `tf.GradientTape` and a central-difference gradient check.
- `forward` + `backward` + an update rule train a network with no library: the moons are
  separated by a boundary made of ReLU pieces.
- Adam keeps running averages of each parameter's gradient and squared gradient, so every
  parameter takes steps of about $\alpha$ in a consistent direction and shorter ones when it
  oscillates. Its advantage is robustness: on $w_1^2 + 10w_2^2$ gradient descent converges only
  for $\alpha < 0.1$, Adam for every $\alpha$ from $0.03$ to $3$.

### Exercises to hand in

1. **Wider and deeper.** Extend `forward` and `backward` to a $2 \to 8 \to 8 \to 1$ network
   (two ReLU hidden layers). Pass the gradient check, then train it on the moons with Adam.
   Compare the boundary with the one-hidden-layer network's.
2. **Softmax back-prop.** For a softmax output with cross-entropy loss,
   $\partial L/\partial z_j = a_j - [j = y]$. Verify this numerically for
   $\mathbf{z} = [2, 1, 0.1]$ and each $y$, using the `log_softmax` of Part 2.
3. **Zero initialisation.** Train the moons network from all-zero weights. Show that all 8
   hidden units stay identical, and explain why from the back-propagation formulas.
4. **Learning-rate sensitivity.** Train the moons network with gradient descent for
   $\alpha \in \{0.01, 0.1, 0.5, 2, 5\}$ and with Adam for
   $\alpha \in \{0.0001, 0.001, 0.01, 0.1, 0.5\}$. Plot final cost against $\alpha$ for both on
   a log axis. Which optimiser is less sensitive to the choice?
5. **Cost of back-prop.** Time `backward` against a full central-difference gradient for a
   $2 \to n \to 1$ network with $n = 4, 16, 64, 256$. Plot both times against the number of
   parameters $P$, and relate the result to the lecture's $N + P$ versus $N \times P$.

### Next

**Part 4 &mdash; Practice lab:** a ten-class handwritten-digit classifier, built and trained in
TensorFlow, then reproduced in NumPy from its trained weights.